In [1]:
# CozyDay AI Prototype — portable model loader
# Weights are cached in the Hugging Face default cache (or HF_HOME if set externally).
# No machine-specific cache path is hard-coded.

import os
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model(model_id):
    """Download/load a Hugging Face model using a portable cache location.

    On CUDA, float16 reduces VRAM use. This is NOT GGUF quantization.
    A large model may still exceed available memory and fail.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    kwargs = {"torch_dtype": torch.float16} if torch.cuda.is_available() else {}
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if torch.cuda.is_available():
        model = model.to('cuda')
    model.eval()
    return tokenizer, model

/opt/anaconda3/envs/ai_llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



# CozyDay AI Prototype

## 1. Research Objective

This project evaluates 15 Hugging Face language models for a daily planning assistant integrated into the CozyDay Django application.

The AI assistant should help users:
- Summarize daily PlanItems.
- Identify high-priority tasks.
- Suggest a realistic order for completing tasks.
- Provide concise, actionable planning recommendations.

## 2. Evaluation Design

All 15 models will be evaluated using the same fixed CozyDay test inputs and expected response format.

Models will be organized into five parameter-size categories, with three models in each category.

### Evaluation Criteria

Each model will be evaluated on:

1. **Instruction Adherence (0–5):** Does the model follow the requested instructions?
2. **Output Quality (0–5):** Are its recommendations relevant, clear, and useful?
3. **Formatting Consistency (0–5):** Does the response follow the required format?
4. **Response Time (seconds):** How long does generation take?
5. **Compute Requirements:** Memory usage, hardware requirements, and loading success or failure.

Higher scores indicate better performance for the three qualitative criteria. Response time and compute requirements will be recorded separately.

## 3. Team Workflow

- Models 1–5: Lin — Research Foundation
- Models 6–10: Rishabh — Prompt Evaluation
- Models 11–15: Shivani — Local LLM Integration

The same evaluation framework and test inputs will be reused across all three stages.


## Setup (clean environment)

Install project requirements from the repository root:
`python -m pip install -r requirements.txt`.

For Google Colab, select a GPU runtime and install the notebook-specific packages with `%pip install torch transformers accelerate pandas`. Model 5 uses `device_map="auto"` with GPU/CPU offloading and therefore requires `accelerate`.

The notebook uses the Hugging Face default cache or an externally configured `HF_HOME`. No Windows-only cache path is required. Model weights and caches must not be committed to Git.

The current official Model 4?5 baseline CSVs contain FP16 Hugging Face Transformers results from Google Colab, not the earlier GGUF/llama.cpp experiments. Model 5 used GPU/CPU offloading, so inference times are not directly comparable across environments.

## File paths and evaluation schema

Open the notebook from `llm-test/` or run it from the repository root; both are supported.

The official `results/model_1_results.csv` through `results/model_5_results.csv` files contain 15 scored test records (three per model) using the shared rubric and consistent CSV columns.

Models 4?5 use official FP16 Transformers inference results. Their earlier GGUF/llama.cpp experiments are historical only and are not included in the current baseline CSVs.

The optional runners are disabled by default (`RUN_BASELINES = False` and `RUN_COLAB_FP16 = False`). Existing saved CSV results can be inspected without rerunning or downloading large models. Model 5 reproduction requires `accelerate` for `device_map="auto"` and CPU offloading.

In [3]:
# Portable results directory and consistent benchmark columns
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / 'results' / 'research_matrix.csv').exists():
    RESULTS_DIR = HERE / 'results'
elif (HERE / 'llm-test' / 'results' / 'research_matrix.csv').exists():
    RESULTS_DIR = HERE / 'llm-test' / 'results'
else:
    raise FileNotFoundError('Run from llm-test/ or repository root with llm-test/results present.')

BASELINE_COLUMNS = [
    'Model', 'Test ID', 'Status', 'Response', 'Response Time (s)',
    'Generated Tokens', 'Reached Token Limit', 'Error',
    'Instruction Adherence', 'Output Quality', 'Formatting Consistency',
    'Total Score', 'Quantization', 'Backend', 'Prompt Mode', 'Evaluation Notes',
]
print('Reading saved data from:', RESULTS_DIR)


Reading saved data from: /Users/shivanibeniwal/Downloads/Semester_3/IS_490/Project_1/3_CozyDay/llm-test/results


In [5]:

import pandas as pd

categories = [
    "Tiny (<0.5B)",
    "Small (0.5B-1.5B)",
    "Medium (>1.5B-3B)",
    "Large (>3B-7B)",
    "Very Large (>7B)"
]

rows = []

for round_number in range(3):
    for category in categories:
        rows.append({
            "Model No.": len(rows) + 1,
            "Category": category,
            "Model Name": "",
            "Hugging Face URL": "",
            "Parameters": "",
            "License / Access": "",
            "Hardware Requirements": "",
            "Loading Instructions": "",
            "Test Owner": ["Lin", "Rishabh", "Shivani"][round_number],
            "Status": "Not tested"
        })

model_matrix = pd.DataFrame(rows)

display(model_matrix)


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
0,1,Tiny (<0.5B),,,,,,,Lin,Not tested
1,2,Small (0.5B-1.5B),,,,,,,Lin,Not tested
2,3,Medium (>1.5B-3B),,,,,,,Lin,Not tested
3,4,Large (>3B-7B),,,,,,,Lin,Not tested
4,5,Very Large (>7B),,,,,,,Lin,Not tested
5,6,Tiny (<0.5B),,,,,,,Rishabh,Not tested
6,7,Small (0.5B-1.5B),,,,,,,Rishabh,Not tested
7,8,Medium (>1.5B-3B),,,,,,,Rishabh,Not tested
8,9,Large (>3B-7B),,,,,,,Rishabh,Not tested
9,10,Very Large (>7B),,,,,,,Rishabh,Not tested


In [6]:

# Stage 1: Research Matrix — Tiny Models

tiny_models = [
    (0, "HuggingFaceTB/SmolLM2-135M-Instruct", "135M"),
    (5, "HuggingFaceTB/SmolLM-135M-Instruct", "135M"),
    (10, "HuggingFaceTB/SmolLM2-360M-Instruct", "360M")
]

for index, model_id, parameters in tiny_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "Hardware Requirements"] = (
        "CPU; estimate RAM needs before testing"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM"
    )

display(model_matrix.iloc[[0, 5, 10]])


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
0,1,Tiny (<0.5B),HuggingFaceTB/SmolLM2-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM2-1...,135M,,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Lin,Not tested
5,6,Tiny (<0.5B),HuggingFaceTB/SmolLM-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM-13...,135M,,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Rishabh,Not tested
10,11,Tiny (<0.5B),HuggingFaceTB/SmolLM2-360M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM2-3...,360M,,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Shivani,Not tested


In [7]:

# Stage 1: Research Matrix — Small Models

small_models = [
    (1, "Qwen/Qwen2.5-0.5B-Instruct", "0.5B"),
    (6, "Qwen/Qwen3-0.6B", "0.6B"),
    (11, "Qwen/Qwen2.5-1.5B-Instruct", "1.5B")
]

for index, model_id, parameters in small_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = "Apache-2.0; public"
    model_matrix.loc[index, "Hardware Requirements"] = (
        "CPU possible; RAM requirement to be benchmarked"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM"
    )

display(model_matrix.iloc[[1, 6, 11]])


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
1,2,Small (0.5B-1.5B),Qwen/Qwen2.5-0.5B-Instruct,https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct,0.5B,Apache-2.0; public,CPU possible; RAM requirement to be benchmarked,AutoTokenizer + AutoModelForCausalLM,Lin,Not tested
6,7,Small (0.5B-1.5B),Qwen/Qwen3-0.6B,https://huggingface.co/Qwen/Qwen3-0.6B,0.6B,Apache-2.0; public,CPU possible; RAM requirement to be benchmarked,AutoTokenizer + AutoModelForCausalLM,Rishabh,Not tested
11,12,Small (0.5B-1.5B),Qwen/Qwen2.5-1.5B-Instruct,https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct,1.5B,Apache-2.0; public,CPU possible; RAM requirement to be benchmarked,AutoTokenizer + AutoModelForCausalLM,Shivani,Not tested


In [8]:

# Stage 1: Research Matrix — Medium Models

medium_models = [
    (2, "Qwen/Qwen3-1.7B", "1.7B",
     "Apache-2.0; public"),
    (7, "google/gemma-2-2b-it", "~2B",
     "Gemma license; acceptance required"),
    (12, "Qwen/Qwen2.5-3B-Instruct", "~3B",
     "Qwen Research License; public")
]

for index, model_id, parameters, license_info in medium_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = license_info
    model_matrix.loc[index, "Hardware Requirements"] = (
        "Several GB RAM required; CPU may be slow"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "check model card and access requirements"
    )

display(model_matrix.iloc[[2, 7, 12]])


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
2,3,Medium (>1.5B-3B),Qwen/Qwen3-1.7B,https://huggingface.co/Qwen/Qwen3-1.7B,1.7B,Apache-2.0; public,Several GB RAM required; CPU may be slow,AutoTokenizer + AutoModelForCausalLM; check mo...,Lin,Not tested
7,8,Medium (>1.5B-3B),google/gemma-2-2b-it,https://huggingface.co/google/gemma-2-2b-it,~2B,Gemma license; acceptance required,Several GB RAM required; CPU may be slow,AutoTokenizer + AutoModelForCausalLM; check mo...,Rishabh,Not tested
12,13,Medium (>1.5B-3B),Qwen/Qwen2.5-3B-Instruct,https://huggingface.co/Qwen/Qwen2.5-3B-Instruct,~3B,Qwen Research License; public,Several GB RAM required; CPU may be slow,AutoTokenizer + AutoModelForCausalLM; check mo...,Shivani,Not tested


In [9]:

# Stage 1: Research Matrix — Large Models

large_models = [
    (3, "microsoft/Phi-3.5-mini-instruct", "3.8B",
     "MIT; public"),
    (8, "Qwen/Qwen3-4B", "4B",
     "Apache-2.0; public"),
    (13, "Qwen/Qwen2.5-7B-Instruct", "7B",
     "Apache-2.0; public")
]

for index, model_id, parameters, license_info in large_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = license_info
    model_matrix.loc[index, "Hardware Requirements"] = (
        "Approx. 8-14 GB FP16 weights; "
        "additional RAM needed for inference"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "review model-specific loading guidance"
    )

display(model_matrix.iloc[[3, 8, 13]])


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
3,4,Large (>3B-7B),microsoft/Phi-3.5-mini-instruct,https://huggingface.co/microsoft/Phi-3.5-mini-...,3.8B,MIT; public,Approx. 8-14 GB FP16 weights; additional RAM n...,AutoTokenizer + AutoModelForCausalLM; review m...,Lin,Not tested
8,9,Large (>3B-7B),Qwen/Qwen3-4B,https://huggingface.co/Qwen/Qwen3-4B,4B,Apache-2.0; public,Approx. 8-14 GB FP16 weights; additional RAM n...,AutoTokenizer + AutoModelForCausalLM; review m...,Rishabh,Not tested
13,14,Large (>3B-7B),Qwen/Qwen2.5-7B-Instruct,https://huggingface.co/Qwen/Qwen2.5-7B-Instruct,7B,Apache-2.0; public,Approx. 8-14 GB FP16 weights; additional RAM n...,AutoTokenizer + AutoModelForCausalLM; review m...,Shivani,Not tested


In [10]:

# Stage 1: Research Matrix — Very Large Models

very_large_models = [
    (4, "Qwen/Qwen3-8B", "8.2B", 8.2),
    (9, "mistralai/Mistral-Nemo-Instruct-2407", "12B", 12.0),
    (14, "Qwen/Qwen2.5-14B-Instruct", "14.7B", 14.7)
]

for index, model_id, parameters, billions in very_large_models:
    model_matrix.loc[index, "Model Name"] = model_id
    model_matrix.loc[index, "Hugging Face URL"] = (
        "https://huggingface.co/" + model_id
    )
    model_matrix.loc[index, "Parameters"] = parameters
    model_matrix.loc[index, "License / Access"] = "Apache-2.0; public"
    model_matrix.loc[index, "Hardware Requirements"] = (
        f"FP16 weights ~{billions * 2:.1f} GB; "
        "additional RAM needed; check feasibility before loading"
    )
    model_matrix.loc[index, "Loading Instructions"] = (
        "AutoTokenizer + AutoModelForCausalLM; "
        "follow model-specific chat template"
    )

display(model_matrix.iloc[[4, 9, 14]])


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
4,5,Very Large (>7B),Qwen/Qwen3-8B,https://huggingface.co/Qwen/Qwen3-8B,8.2B,Apache-2.0; public,FP16 weights ~16.4 GB; additional RAM needed; ...,AutoTokenizer + AutoModelForCausalLM; follow m...,Lin,Not tested
9,10,Very Large (>7B),mistralai/Mistral-Nemo-Instruct-2407,https://huggingface.co/mistralai/Mistral-Nemo-...,12B,Apache-2.0; public,FP16 weights ~24.0 GB; additional RAM needed; ...,AutoTokenizer + AutoModelForCausalLM; follow m...,Rishabh,Not tested
14,15,Very Large (>7B),Qwen/Qwen2.5-14B-Instruct,https://huggingface.co/Qwen/Qwen2.5-14B-Instruct,14.7B,Apache-2.0; public,FP16 weights ~29.4 GB; additional RAM needed; ...,AutoTokenizer + AutoModelForCausalLM; follow m...,Shivani,Not tested


In [11]:

# Complete missing license information for Tiny models

tiny_indices = [0, 5, 10]

model_matrix.loc[
    tiny_indices, "License / Access"
] = "Apache-2.0; public"

# Verify all required fields
required_columns = [
    "Model Name",
    "Hugging Face URL",
    "Parameters",
    "License / Access",
    "Hardware Requirements",
    "Loading Instructions"
]

missing = model_matrix[required_columns].isna() | (
    model_matrix[required_columns].astype(str).apply(
        lambda col: col.str.strip().eq("")
    )
)

print("Total models:", len(model_matrix))
print("Missing required fields:", int(missing.sum().sum()))
print("Unique models:", model_matrix["Model Name"].nunique())

display(model_matrix.iloc[[0, 5, 10]])


Total models: 15
Missing required fields: 0
Unique models: 15


,Model No.,Category,Model Name,Hugging Face URL,Parameters,License / Access,Hardware Requirements,Loading Instructions,Test Owner,Status
0,1,Tiny (<0.5B),HuggingFaceTB/SmolLM2-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM2-1...,135M,Apache-2.0; public,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Lin,Not tested
5,6,Tiny (<0.5B),HuggingFaceTB/SmolLM-135M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM-13...,135M,Apache-2.0; public,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Rishabh,Not tested
10,11,Tiny (<0.5B),HuggingFaceTB/SmolLM2-360M-Instruct,https://huggingface.co/HuggingFaceTB/SmolLM2-3...,360M,Apache-2.0; public,CPU; estimate RAM needs before testing,AutoTokenizer + AutoModelForCausalLM,Shivani,Not tested


In [12]:

# Stage 1: Fixed CozyDay Test Inputs
# These inputs must remain unchanged across all 15 models.

test_cases = [
    {
        "test_id": "T1",
        "name": "Simple Priority Planning",
        "input": (
            "I need to finish a report due tomorrow, "
            "buy groceries, and clean my room. "
            "Which task should I prioritize and why?"
        )
    },
    {
        "test_id": "T2",
        "name": "Deadline and Time Management",
        "input": (
            "I have a meeting from 2 PM to 3 PM, "
            "an assignment due at 5 PM that needs 2 hours, "
            "and a 30-minute workout. "
            "Suggest a realistic schedule starting at 12 PM."
        )
    },
    {
        "test_id": "T3",
        "name": "Busy Day Planning",
        "input": (
            "Today I need to study for 3 hours, "
            "prepare a presentation for 2 hours, "
            "reply to emails for 30 minutes, "
            "and attend class from 1 PM to 3 PM. "
            "I am available from 9 AM to 6 PM. "
            "Help me plan my day with short breaks."
        )
    }
]

for case in test_cases:
    print(f"\n{case['test_id']}: {case['name']}")
    print(case["input"])



T1: Simple Priority Planning
I need to finish a report due tomorrow, buy groceries, and clean my room. Which task should I prioritize and why?

T2: Deadline and Time Management
I have a meeting from 2 PM to 3 PM, an assignment due at 5 PM that needs 2 hours, and a 30-minute workout. Suggest a realistic schedule starting at 12 PM.

T3: Busy Day Planning
Today I need to study for 3 hours, prepare a presentation for 2 hours, reply to emails for 30 minutes, and attend class from 1 PM to 3 PM. I am available from 9 AM to 6 PM. Help me plan my day with short breaks.



## 4. Scoring Rubric

All models will be evaluated on the same three CozyDay test cases (T1, T2, T3).

### Required Response Format

Each model should produce:

**Priority:** The most urgent task or goal.

**Schedule:** A realistic ordered plan with time slots when appropriate.

**Reason:** A short explanation of the recommended order.

The response should be concise, actionable, and avoid inventing deadlines or task constraints.

### Scoring Dimensions

Each dimension is scored from 0 to 5.

| Score | Instruction Adherence | Output Quality | Formatting Consistency |
|---|---|---|---|
| 5 | Follows all instructions | Accurate, realistic, actionable | All three sections present and clear |
| 4 | Minor instruction omission | Useful with minor issues | Minor formatting deviations |
| 3 | Follows some instructions | Partially useful or incomplete | Two sections correctly structured |
| 2 | Follows few instructions | Major omissions or planning errors | One section correctly structured |
| 1 | Barely follows instructions | Mostly irrelevant or impractical | Unstructured response |
| 0 | Does not follow instructions | No useful answer | No requested structure |

### Overall Score

Total Score = Instruction Adherence + Output Quality + Formatting Consistency

Maximum score per test case: 15 points.

Maximum score across three test cases: 45 points.

### Additional Metrics

- **Response Time:** Seconds required for generation.
- **Compute Requirements:** Device, estimated RAM usage, and loading success/failure.
- **Error Notes:** Model access restrictions, out-of-memory errors, timeouts, or other failures.

Models that cannot load or generate output will be marked as **Not evaluated** for qualitative scores. Resource failures must be documented rather than assigned an artificial score of zero.


## Stage 1 — Manual scoring audit (official FP16 Transformers reruns)

The original rubric gives **0–5 each** for Instruction Adherence (IA), Output Quality (OQ), and Formatting Consistency (FC); each test is out of **15 points**. Generation speed and hardware are recorded separately, not scored. Raw responses remain unchanged.

| Model | Case | IA | OQ | FC | Total | Evidence |
|---|---|---:|---:|---:|---:|---|
| Phi-3.5-mini | T1 | 4 | 4 | 5 | **13** | Correct task priority; assumes report needs 2–3 hours without user-supplied estimate; all headings clear |
| Phi-3.5-mini | T2 | 2 | 2 | 4 | **8** | Fixed 2–3 PM meeting reduced to 2:30–3; extra assignment time; explanation truncated at 256 tokens |
| Phi-3.5-mini | T3 | 2 | 2 | 4 | **8** | Study 3h45m rather than 3h; 15m break mislabeled as 30m; explanation truncated at 256 tokens |
| Qwen3-8B | T1 | 5 | 5 | 5 | **15** | Appropriate report-first priority with grocery/cleaning follow-up, all three headings present |
| Qwen3-8B | T2 | 2 | 2 | 5 | **9** | Only 1h assignment work + 30m finalization versus requested 2h; redundant preparation slot; format intact |
| Qwen3-8B | T3 | 3 | 2 | 5 | **10** | Explicit study 2h versus 3h, presentation 1h45m versus 2h; unallocated time is ambiguous; format intact |

**Totals:** Phi-3.5-mini **29/45**, Qwen3-8B **34/45**. Judgments apply only to the recorded rerun outputs, not the earlier GGUF outputs. Qwen3-8B used `enable_thinking=False`; its GPU/CPU offloading latency must not be interpreted as a model-only speed comparison.


In [13]:

# Stage 1: Shared Baseline Prompt
# Reuse this exact instruction for all 15 models.

SYSTEM_INSTRUCTION = (
    "You are CozyDay, a helpful daily planning assistant. "
    "Use only the information provided by the user. "
    "Give realistic, concise, actionable planning advice. "
    "Do not invent deadlines or appointments. "
    "Use exactly these three headings:\n"
    "Priority:\n"
    "Schedule:\n"
    "Reason:"
)

def build_prompt(tokenizer, user_input):
    """
    Format a CozyDay request using the model's chat template
    when available, or a plain-text fallback.
    """
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_input}
    ]

    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except (ValueError, TypeError):
            # Some models do not support a system role.
            pass

    return (
        f"Instructions:\n{SYSTEM_INSTRUCTION}\n\n"
        f"User request:\n{user_input}\n\n"
        "Assistant response:\n"
    )

print("Baseline prompt is ready.")


Baseline prompt is ready.


In [14]:
# Stage 1: Improved Baseline Test Harness
# Shared configuration for all 15 models

MAX_NEW_TOKENS = 256

def test_model_v2(model_id, test_cases):
    import time
    import gc
    import torch
    import pandas as pd

    results = []
    tokenizer = None
    model = None

    try:
        tokenizer, model = load_model(model_id)
        model.eval()

        for case in test_cases:
            try:
                prompt = build_prompt(tokenizer, case["input"])
                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=1024
                )

                inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

                start = time.perf_counter()

                with torch.inference_mode():
                    output = model.generate(
                        **inputs,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id
                    )

                elapsed = time.perf_counter() - start
                new_tokens = output[0][
                    inputs["input_ids"].shape[1]:
                ]
                response = tokenizer.decode(
                    new_tokens,
                    skip_special_tokens=True
                )

                results.append({
                    "Model": model_id,
                    "Test ID": case["test_id"],
                    "Status": "Success",
                    "Response": response,
                    "Response Time (s)": round(elapsed, 2),
                    "Generated Tokens": len(new_tokens),
                    "Reached Token Limit": (
                        len(new_tokens) >= MAX_NEW_TOKENS
                    ),
                    "Error": ""
                })

            except Exception as error:
                results.append({
                    "Model": model_id,
                    "Test ID": case["test_id"],
                    "Status": "Failed",
                    "Response": "",
                    "Response Time (s)": None,
                    "Generated Tokens": None,
                    "Reached Token Limit": None,
                    "Error": str(error)
                })

    except Exception as error:
        for case in test_cases:
            results.append({
                "Model": model_id,
                "Test ID": case["test_id"],
                "Status": "Load failed",
                "Response": "",
                "Response Time (s)": None,
                "Generated Tokens": None,
                "Reached Token Limit": None,
                "Error": str(error)
            })

    finally:
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

print("Improved baseline harness ready.")
print("Generation limit:", MAX_NEW_TOKENS)

Improved baseline harness ready.
Generation limit: 256


In [15]:
# Explicit reproducible benchmark runner for Models 1–5.
# CAUTION: This is opt-in to avoid automatically downloading multi-GB models on Run All.
# Set RUN_BASELINES = True ONLY on a machine with sufficient RAM/VRAM and disk space.
# Scores are intentionally left blank: manual scoring must be done using the shared rubric.

from pathlib import Path
import pandas as pd

MODEL_IDS_STAGE1 = {
    1: 'HuggingFaceTB/SmolLM2-135M-Instruct',
    2: 'Qwen/Qwen2.5-0.5B-Instruct',
    3: 'Qwen/Qwen3-1.7B',
    4: 'microsoft/Phi-3.5-mini-instruct',
    5: 'Qwen/Qwen3-8B',
}

RUN_BASELINES = False
MODELS_TO_RUN = [1, 2, 3, 4, 5]  # Edit to run one model at a time when necessary.

# All five model slots can be rerun independently; use the shared test cases.
# Scored official FP16 baselines must not be overwritten by reruns.
if RUN_BASELINES:
    for model_no in MODELS_TO_RUN:
        model_id = MODEL_IDS_STAGE1[model_no]
        print(f'Transformers baseline: Model {model_no} — {model_id}')
        evaluated = test_model_v2(model_id, test_cases)
        for column in ['Instruction Adherence', 'Output Quality', 'Formatting Consistency', 'Total Score']:
            evaluated[column] = pd.NA
        evaluated['Quantization'] = 'No GGUF quantization (Transformers)'
        evaluated['Backend'] = 'Transformers'
        evaluated['Prompt Mode'] = 'Baseline (SYSTEM_INSTRUCTION)'
        evaluated['Evaluation Notes'] = 'Pending manual scoring; see rubric.'
        output = RESULTS_DIR / f'model_{model_no}_transformers_rerun.csv'
        output.parent.mkdir(parents=True, exist_ok=True)
        # Never overwrite original manually scored GGUF or Transformers results.
        if output.exists():
            raise FileExistsError(f'{output} already exists; rename or archive it first.')
        evaluated = evaluated.reindex(columns=BASELINE_COLUMNS)
        evaluated.to_csv(output, index=False)
        print('Saved:', output)
else:
    print('Baselines are not automatically rerun. Set RUN_BASELINES=True to execute and save.')

Baselines are not automatically rerun. Set RUN_BASELINES=True to execute and save.


In [16]:
# Optional: reproduce the official FP16 Colab baselines for Models 4 or 5.
# Run ONLY on Colab with suitable GPU/CPU memory; set RUN_COLAB_FP16=True deliberately.
# Requires torch, transformers, accelerate. See project requirements.txt.
RUN_COLAB_FP16 = False
COLAB_MODEL_NO = 4  # set to 5 for Qwen3-8B

if RUN_COLAB_FP16:
    import torch, time, gc, pandas as pd
    from transformers import AutoTokenizer, AutoModelForCausalLM
    if not torch.cuda.is_available():
        raise RuntimeError('This reproducibility cell requires a CUDA GPU in Colab.')
    selected = MODEL_IDS_STAGE1[COLAB_MODEL_NO]
    tok = AutoTokenizer.from_pretrained(selected, trust_remote_code=False)
    kwargs = dict(dtype=torch.float16, trust_remote_code=False, low_cpu_mem_usage=True)
    if COLAB_MODEL_NO == 4:
        kwargs['attn_implementation'] = 'eager'
    elif COLAB_MODEL_NO == 5:
        kwargs.update(device_map='auto', max_memory={0:'12GiB','cpu':'7GiB'}, offload_folder='/content/qwen3_offload')
    else:
        raise ValueError('Select Model 4 or 5.')
    mdl = AutoModelForCausalLM.from_pretrained(selected, **kwargs)
    if COLAB_MODEL_NO == 4: mdl = mdl.to('cuda')
    mdl.eval()
    rows=[]
    for case in test_cases:
        opts = {'enable_thinking':False} if COLAB_MODEL_NO == 5 else {}
        msgs = [{'role':'system','content':SYSTEM_INSTRUCTION},{'role':'user','content':case['input']}]
        prompt = tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,**opts)
        tokens = tok(prompt,return_tensors='pt',truncation=True,max_length=1024).to('cuda')
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.inference_mode():
            output=mdl.generate(**tokens,max_new_tokens=256,do_sample=False,pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        new_tokens=output[0][tokens['input_ids'].shape[1]:]
        rows.append({'Model':selected,'Test ID':case['test_id'],'Status':'Success',
                     'Response':tok.decode(new_tokens,skip_special_tokens=True),
                     'Response Time (s)':round(elapsed,2),'Generated Tokens':len(new_tokens),
                     'Reached Token Limit':len(new_tokens)>=256,'Error':'',
                     'Instruction Adherence':pd.NA,'Output Quality':pd.NA,
                     'Formatting Consistency':pd.NA,'Total Score':pd.NA,
                     'Quantization':'None (FP16)',
                     'Backend':'Transformers / Colab T4 '+('GPU' if COLAB_MODEL_NO==4 else 'GPU + CPU offload'),
                     'Prompt Mode':'Baseline; thinking disabled' if COLAB_MODEL_NO==5 else 'Baseline',
                     'Evaluation Notes':'Pending manual scoring'})
        print(case['test_id'], round(elapsed,2),'seconds')
    path=RESULTS_DIR/f'model_{COLAB_MODEL_NO}_transformers_new_run.csv'
    if path.exists():raise FileExistsError(path)
    pd.DataFrame(rows,columns=BASELINE_COLUMNS).to_csv(path,index=False)
    print('Saved:',path)
    del mdl; gc.collect(); torch.cuda.empty_cache()
else:
    print('Official FP16 reproduction disabled (set RUN_COLAB_FP16=True in a suitable environment).')


Official FP16 reproduction disabled (set RUN_COLAB_FP16=True in a suitable environment).


In [17]:

# Load all saved CozyDay research and baseline results
# No model downloads or inference are performed here.

from pathlib import Path
import pandas as pd

results_dir = RESULTS_DIR

model_matrix = pd.read_csv(
    results_dir / "research_matrix.csv"
)

model_results = {}

for model_no in range(1, 6):
    model_results[model_no] = pd.read_csv(
        results_dir / f"model_{model_no}_results.csv"
    )

hardware_assessment = pd.read_csv(
    results_dir / "model_4_5_hardware_assessment.csv"
)

print("Research models:", len(model_matrix))
print("Baseline models loaded:", len(model_results))
print("Baseline test records:", sum(
    len(df) for df in model_results.values()
))


Research models: 15
Baseline models loaded: 5
Baseline test records: 15


In [18]:

# Stage 1 evaluation summary: Models 1–5

summary_rows = []

for model_no, df in model_results.items():
    summary_rows.append({
        "Model": model_no,
        "Test Cases": len(df),
        "Scored Cases": int(df["Total Score"].notna().sum()),
        "Total Score / 45": df["Total Score"].sum()
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)


,Model,Test Cases,Scored Cases,Total Score / 45
0,1,3,3,6.0
1,2,3,3,13.0
2,3,3,3,12.0
3,4,3,3,29.0
4,5,3,3,34.0


## Stage 1 — Results and Team Handoff

**Completed:** 15-model research matrix; fixed test inputs; shared rubric and Transformers test harness; scored baselines for Models 1–5.

**Actual inference:** Models 1–3 were assessed with Hugging Face Transformers on local CPU. Model 4 (`microsoft/Phi-3.5-mini-instruct`) was rerun in Google Colab using the official **FP16** model and Transformers 4.57.6 on Tesla T4 GPU. Model 5 (`Qwen/Qwen3-8B`) was rerun with the official **FP16** model and Transformers 4.57.6 on Tesla T4 GPU **with CPU offloading**. No GGUF results are used as the current Model 4–5 baseline.

**Scoring review:** Model 4–5 scores were audited case by case against the shared 0–5 × 3 rubric; reasons appear above and in each CSV evaluation note. Raw Colab outputs are retained as local evidence outside the Git repository.

**Timing:** The two Colab runs report synchronized generation time, not download/model loading. CPU offloading for Model 5 produces dramatically different latency. Models 1–3 used a different execution environment. Do not interpret speed as a controlled direct comparison.

**Method details:** Both reruns used the three unchanged T1–T3 inputs, the shared `SYSTEM_INSTRUCTION`, `max_new_tokens=256`, `do_sample=False`, and the official chat templates. For Qwen3-8B, `enable_thinking=False` was explicitly set; this should be stated when comparing to earlier Qwen3 tests.

**Known limitations:** Qwen3-1.7B baseline can include `<think>` text and reach the generation limit. Model 4 T2/T3 hit 256 tokens. Model 5 T2/T3 contain scheduling errors. Save raw responses without editing them. The optional baseline runner does **not** rerun large models automatically.

**Next stage:** Rishabh handles model slots 6–10 and zero/one/few/many-shot prompting. Shivani handles 11–15 and local integration. Never commit weights, secrets or caches.


## Stage 2 — Rishabh — Prompt Evaluation (Models 6–10)


In [19]:
# Stage 2: Few-shot example pool (NOT the T1-T3 fixed test cases — separate, held-out examples)

SHOT_EXAMPLES = [
    {
        "input": "I have a dentist appointment at 10 AM, need to walk the dog, and want to read for 30 minutes.",
        "response": (
            "Priority: Dentist appointment at 10 AM\n"
            "Schedule:\n"
            "1. Walk the dog before 9:30 AM.\n"
            "2. Attend the dentist appointment at 10 AM.\n"
            "3. Read for 30 minutes after you return.\n"
            "Reason: The appointment is time-fixed, so the flexible tasks are placed around it."
        ),
    },
    {
        "input": "I need to pay bills, do laundry, and call my parents, but I only have 2 hours free tonight.",
        "response": (
            "Priority: Pay bills\n"
            "Schedule:\n"
            "1. Pay bills (20 minutes).\n"
            "2. Start laundry, then call your parents while it runs (45 minutes).\n"
            "3. Finish folding laundry (20 minutes).\n"
            "Reason: Bills often have deadlines and are prioritized first; the call is layered into idle laundry time to fit the 2-hour window."
        ),
    },
    {
        "input": "I have a 9 AM flight, need to pack, and want to check in online.",
        "response": (
            "Priority: Check in online\n"
            "Schedule:\n"
            "1. Check in online the night before.\n"
            "2. Pack the night before as well.\n"
            "3. Leave with enough buffer for a 9 AM flight.\n"
            "Reason: Check-in and packing should happen ahead of time since the flight time itself is fixed and non-negotiable."
        ),
    },
    {
        "input": "I need to exercise, write a journal entry, and prep lunch for tomorrow, free from 7 PM to 9 PM.",
        "response": (
            "Priority: Exercise\n"
            "Schedule:\n"
            "1. Exercise from 7:00-7:45 PM.\n"
            "2. Prep lunch from 7:45-8:15 PM.\n"
            "3. Write journal entry from 8:15-8:30 PM.\n"
            "Reason: Exercise is placed first while energy is higher; shorter tasks follow in the remaining window."
        ),
    },
    {
        "input": "I have a doctor's appointment at 3 PM, need to finish a work report, and want to water my plants.",
        "response": (
            "Priority: Finish the work report\n"
            "Schedule:\n"
            "1. Work on the report in the morning.\n"
            "2. Water the plants on a short break.\n"
            "3. Leave in time for the 3 PM doctor's appointment.\n"
            "Reason: The report has no fixed time and benefits from focused morning work; the appointment anchors the afternoon."
        ),
    },
]

def build_prompt_shots(tokenizer, user_input, n_shots=0):
    """Zero/one/few/many-shot prompt builder. n_shots draws from SHOT_EXAMPLES, never from test_cases.

    Some chat templates (e.g. Gemma) reject a 'system' role message outright
    ("System role not supported"). We first try the normal system-role form;
    if the template rejects it for ANY reason, we retry by folding the system
    instruction into the first user turn instead, which Gemma's template accepts.
    Only if both attempts fail do we fall back to plain, non-chat-template text.

    enable_thinking=False is passed to apply_chat_template on every attempt. Qwen3's
    chat template defaults to emitting a visible <think>...</think> reasoning block
    before any answer, which previously consumed the entire max_new_tokens budget on
    Models 7 and 9 before they produced a Priority/Schedule/Reason response (see PR
    review). Templates that don't define an `enable_thinking` variable (Gemma, SmolLM,
    Mistral) simply ignore the extra kwarg, so this is safe across all five models.
    """
    examples = SHOT_EXAMPLES[:n_shots]

    def make_messages(fold_system_into_first_user):
        messages = []
        if not fold_system_into_first_user:
            messages.append({"role": "system", "content": SYSTEM_INSTRUCTION})
        folded = False
        for ex in examples:
            content = ex["input"]
            if fold_system_into_first_user and not folded:
                content = f"{SYSTEM_INSTRUCTION}\n\n{content}"
                folded = True
            messages.append({"role": "user", "content": content})
            messages.append({"role": "assistant", "content": ex["response"]})
        final_content = user_input
        if fold_system_into_first_user and not folded:
            final_content = f"{SYSTEM_INSTRUCTION}\n\n{user_input}"
        messages.append({"role": "user", "content": final_content})
        return messages

    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(
                make_messages(fold_system_into_first_user=False),
                tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
        except Exception:
            try:
                return tokenizer.apply_chat_template(
                    make_messages(fold_system_into_first_user=True),
                    tokenize=False, add_generation_prompt=True,
                    enable_thinking=False,
                )
            except Exception:
                pass

    parts = [f"Instructions:\n{SYSTEM_INSTRUCTION}\n"]
    for ex in examples:
        parts.append(f"User request:\n{ex['input']}\n\nAssistant response:\n{ex['response']}\n")
    parts.append(f"User request:\n{user_input}\n\nAssistant response:\n")
    return "\n".join(parts)

PROMPT_MODES = {"Zero-shot": 0, "One-shot": 1, "Few-shot": 3, "Many-shot": 5}
print("Shot pool and prompt builder ready (Gemma system-role fallback + enable_thinking=False for Qwen3). Modes:", PROMPT_MODES)


Shot pool and prompt builder ready (Gemma system-role fallback + enable_thinking=False for Qwen3). Modes: {'Zero-shot': 0, 'One-shot': 1, 'Few-shot': 3, 'Many-shot': 5}


In [20]:
# Stage 2: Prompt-method comparison runner (mirrors Lin's test_model_v2, adds Prompt Mode)

def test_model_prompt_modes(model_id, test_cases, backend_label="Transformers (local CPU)"):
    import time, gc, torch, pandas as pd

    results = []
    tokenizer, model = load_model(model_id)
    model.eval()

    try:
        for mode_name, n_shots in PROMPT_MODES.items():
            for case in test_cases:
                try:
                    prompt = build_prompt_shots(tokenizer, case["input"], n_shots=n_shots)
                    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
                    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

                    start = time.perf_counter()
                    with torch.inference_mode():
                        output = model.generate(
                            **inputs, max_new_tokens=256, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id
                        )
                    elapsed = time.perf_counter() - start
                    new_tokens = output[0][inputs["input_ids"].shape[1]:]
                    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

                    results.append({
                        "Model": model_id, "Test ID": case["test_id"], "Status": "Success",
                        "Response": response, "Response Time (s)": round(elapsed, 2),
                        "Generated Tokens": len(new_tokens),
                        "Reached Token Limit": len(new_tokens) >= 256, "Error": "",
                        "Instruction Adherence": pd.NA, "Output Quality": pd.NA,
                        "Formatting Consistency": pd.NA, "Total Score": pd.NA,
                        "Quantization": "No GGUF quantization (Transformers)",
                        "Backend": backend_label, "Prompt Mode": mode_name,
                        "Evaluation Notes": "Pending manual scoring; see rubric.",
                    })
                except Exception as error:
                    results.append({
                        "Model": model_id, "Test ID": case["test_id"], "Status": "Failed",
                        "Response": "", "Response Time (s)": None, "Generated Tokens": None,
                        "Reached Token Limit": None, "Error": str(error),
                        "Instruction Adherence": pd.NA, "Output Quality": pd.NA,
                        "Formatting Consistency": pd.NA, "Total Score": pd.NA,
                        "Quantization": "No GGUF quantization (Transformers)",
                        "Backend": backend_label, "Prompt Mode": mode_name,
                        "Evaluation Notes": "Generation failed.",
                    })
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(results, columns=BASELINE_COLUMNS)

print("test_model_prompt_modes ready.")


test_model_prompt_modes ready.


In [21]:
# Stage 2: Local run — start with the two smallest models ONLY.
# Do not add 8/9/10 here until 6 and 7 have completed successfully.

RISHABH_MODELS = {
    6: "HuggingFaceTB/SmolLM-135M-Instruct",
    7: "Qwen/Qwen3-0.6B",
    8: "google/gemma-2-2b-it",                      # gated — accept license on huggingface.co first
    9: "Qwen/Qwen3-4B",
    10: "mistralai/Mistral-Nemo-Instruct-2407",      # 12B — run on Colab, not locally
}

MODELS_TO_RUN_LOCALLY = []  # disabled by default so "Run All" never re-downloads/reruns a model
# and overwrites an already-scored CSV. Set explicitly (e.g. [7]) only when you intend
# to regenerate that model's results.

for model_no in MODELS_TO_RUN_LOCALLY:
    model_id = RISHABH_MODELS[model_no]
    print(f"Running Model {model_no} — {model_id}")
    df = test_model_prompt_modes(model_id, test_cases)
    out = RESULTS_DIR / f"model_{model_no}_results.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    print(df[["Model", "Test ID", "Prompt Mode", "Status", "Response Time (s)"]])



In [22]:
import pandas as pd
df8 = pd.read_csv(RESULTS_DIR / "model_8_results.csv")
print(df8.loc[0, "Error"])

nan


In [23]:
# Stage 2: Colab fallback for Model 10 (Mistral-Nemo-Instruct-2407, 12B) — and 8/9 if they fail locally.
# Open this notebook in Google Colab with a GPU runtime, then set RUN_COLAB_STAGE2 = True.
# Model 10's FP16 weights (~24.5 GB) do not fit a free-tier T4 (14.56 GB VRAM), even with
# CPU offloading (that was tried first and OOM'd). The results in model_10_results.csv were
# produced with 4-bit NF4 quantization via bitsandbytes, forced fully onto the GPU
# (device_map={"": 0} — bitsandbytes 4-bit does not support mixed CPU/GPU dispatch without
# an extra fp32-cpu-offload flag we don't need once the model fits in 4-bit). This cell must
# match that config, or a rerun will not reproduce the recorded results.
# Requires bitsandbytes (see requirements.txt) in addition to torch/transformers/accelerate.

RUN_COLAB_STAGE2 = False
COLAB_MODEL_NO = 10  # set to 8 or 9 if those also need to move to Colab

if RUN_COLAB_STAGE2:
    import torch, time, gc, pandas as pd
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("This cell requires a CUDA GPU in Colab. Runtime > Change runtime type > GPU.")

    selected = RISHABH_MODELS[COLAB_MODEL_NO]
    tok = AutoTokenizer.from_pretrained(selected, trust_remote_code=False)
    if COLAB_MODEL_NO == 10:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        kwargs = dict(quantization_config=bnb_config, device_map={"": 0}, trust_remote_code=False, low_cpu_mem_usage=True)
        quant_label = "4-bit NF4 (bitsandbytes)"
    else:
        kwargs = dict(dtype=torch.float16, trust_remote_code=False, low_cpu_mem_usage=True)
        quant_label = "None (FP16)"
    mdl = AutoModelForCausalLM.from_pretrained(selected, **kwargs)
    if "device_map" not in kwargs:
        mdl = mdl.to("cuda")
    mdl.eval()

    rows = []
    for mode_name, n_shots in PROMPT_MODES.items():
        for case in test_cases:
            prompt = build_prompt_shots(tok, case["input"], n_shots=n_shots)
            tokens = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
            torch.cuda.synchronize(); t0 = time.perf_counter()
            with torch.inference_mode():
                output = mdl.generate(**tokens, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            new_tokens = output[0][tokens["input_ids"].shape[1]:]
            rows.append({
                "Model": selected, "Test ID": case["test_id"], "Status": "Success",
                "Response": tok.decode(new_tokens, skip_special_tokens=True),
                "Response Time (s)": round(elapsed, 2), "Generated Tokens": len(new_tokens),
                "Reached Token Limit": len(new_tokens) >= 256, "Error": "",
                "Instruction Adherence": pd.NA, "Output Quality": pd.NA,
                "Formatting Consistency": pd.NA, "Total Score": pd.NA,
                "Quantization": quant_label, "Backend": "Transformers / Colab GPU",
                "Prompt Mode": mode_name, "Evaluation Notes": "Pending manual scoring",
            })
            print(case["test_id"], mode_name, round(elapsed, 2), "seconds")

    path = RESULTS_DIR / f"model_{COLAB_MODEL_NO}_results.csv"
    pd.DataFrame(rows, columns=BASELINE_COLUMNS).to_csv(path, index=False)
    print("Saved:", path)
else:
    print("Colab run not started. Set RUN_COLAB_STAGE2 = True inside an actual Colab GPU runtime.")


Colab run not started. Set RUN_COLAB_STAGE2 = True inside an actual Colab GPU runtime.


## Stage 2 — Results, Prompt-Method Recommendation, and Team Handoff

**Completed:** All 5 assigned models (slots 6–10) ran successfully across all 4 prompt modes (zero/one/few/many-shot) and all 3 shared test cases (T1–T3) — 60/60 generations, `Status: Success`, zero errors. Models 6–8 ran locally via Transformers (CPU/MPS); Models 9–10 ran in Google Colab on a Tesla T4 GPU (Model 9 in FP16, Model 10 in 4-bit NF4 quantization via `bitsandbytes`, since its FP16 weights do not fit a free-tier T4). All 60 responses have been scored against the Section 4 rubric and written into `llm-test/results/model_6_results.csv` through `model_10_results.csv` (`Instruction Adherence`, `Output Quality`, `Formatting Consistency`, `Total Score`, `Evaluation Notes`).

**Revision note (post-review):** Models 7 and 9 were rerun and rescored after fixing a bug flagged in PR review — `build_prompt_shots` was not passing `enable_thinking=False` to Qwen3's chat template, so both Qwen3 models burned their entire token budget on invisible `<think>` reasoning instead of answering. With the fix, Model 7 went from 47/180 to **118/180** and Model 9 went from 0/180 to **141/180**. The numbers below are the corrected, rerun results.

### Per-model totals (sum across 3 test cases × 4 prompt modes; max 180)

| Model | Total /180 | Notes |
|---|---:|---|
| `mistralai/Mistral-Nemo-Instruct-2407` (4-bit) | **157** | Strongest performer; near-perfect on T1, solid on T2/T3 with occasional time-allocation drift |
| `google/gemma-2-2b-it` | **155** | Essentially tied with Model 10; perfect on T1, minor constraint slips on T2/T3 |
| `Qwen/Qwen3-4B` | **141** | Strong once `enable_thinking=False` is set; perfect on T1 and on T2/zero-shot, recurring error placing the fixed 2-3 PM meeting at 1-3 PM instead |
| `Qwen/Qwen3-0.6B` | 118 | Solid on T1 across all modes; T2 has a recurring fixed-meeting-time error, T3 struggles with exact time allocation |
| `HuggingFaceTB/SmolLM-135M-Instruct` | 25 | Too small to reliably follow the task; best result still only 11/15, most cases 0–6 |

### Prompt-method comparison

Summed score by prompt mode (all 5 models, max 75 per mode):

| Mode | Zero-shot | One-shot | Few-shot | Many-shot |
|---|---:|---:|---:|---:|
| Total | 135 | 149 | **162** | 150 |

**Recommendation: few-shot prompting (3 examples) gave the best results across this model slot**, consistent with the pre-fix finding and now confirmed on real, complete data from all 5 models rather than 2 that were mostly failing. Many-shot performed close behind but not better — extra examples beyond ~3 did not reliably buy more adherence or quality. Zero-shot was consistently the weakest mode: with no example to anchor the `Priority / Schedule / Reason` format, responses were more likely to omit a section, invent unrelated content, or ignore an explicit time constraint.

### Fixed limitation — `enable_thinking` now disabled for Qwen3 models

Qwen3's chat template defaults to emitting a visible `<think>...</think>` reasoning block before any answer. The original `build_prompt_shots` did not set `enable_thinking=False`, so this reasoning block routinely consumed the entire `max_new_tokens=256` budget before Models 7 and 9 could produce a `Priority/Schedule/Reason` answer — most severely on Model 9, which failed on all 12 of 12 runs (0/180). `build_prompt_shots` now passes `enable_thinking=False` on every `apply_chat_template` call; templates that don't define that variable (Gemma, SmolLM, Mistral) simply ignore the extra kwarg, so the fix is safe across all five models. Both Qwen3 models were rerun end-to-end after the fix — see the Revision note above for the before/after scores. With the fix, the models' real weaknesses are now visible and genuine: both Qwen3 models have a recurring pattern of scheduling the fixed 2-3 PM meeting at the wrong time/duration on T2, which is an actual model behavior, not a token-budget artifact.

### Candidate-model handoff for Shivani (Stage 3 — local integration)

- **Recommended:** `google/gemma-2-2b-it` and `mistralai/Mistral-Nemo-Instruct-2407`, scoring 155 and 157 out of 180. Gemma-2-2B is far cheaper to run locally (2B params, FP16, no quantization needed) and is the easier integration target; Mistral-Nemo only reaches this quality at 4-bit quantization and needs a CUDA GPU with `bitsandbytes`, so it is a heavier dependency for local integration unless the target environment already has GPU access.
- **Viable third option:** `Qwen/Qwen3-4B` (141/180) now that `enable_thinking=False` is set — strong on T1 and competitive on T2/T3, with one recurring, documented error (fixed meeting time on T2). Worth considering if Shivani needs a mid-size model with room to grow, but **must** use `enable_thinking=False` in any chat-template call — forgetting it reproduces the original failure.
- **Not recommended:** `Qwen/Qwen3-0.6B` (118/180) — usable but the weakest of the three enable_thinking-fixed models, with the same recurring meeting-time error and less consistent T3 handling.
- **Not recommended:** `HuggingFaceTB/SmolLM-135M-Instruct` (25/180) — too small for this task even with many-shot prompting.
- **Prompt mode:** use **few-shot (3 examples)** as the default prompting strategy; it outperformed zero/one/many-shot in the aggregate.

**Timing / compute:** Models 6–8 ran on local CPU/MPS (Model 8's first three generations took 500–1600s due to memory pressure loading a 3B-class model in FP32 on 16GB of RAM; later generations were 12–40s once weights were cached). Models 9–10 ran on a free-tier Colab T4 GPU; Model 9 in FP16 (5–16s per generation after the enable_thinking fix, versus hitting the 256-token ceiling every time beforehand), Model 10 required 4-bit NF4 quantization and a forced single-GPU `device_map` to fit in 14.5GB VRAM. Do not interpret these times as a controlled speed comparison across models — execution environment differs (local CPU vs. Colab GPU vs. quantization).

**Known limitations:** Raw responses were saved unedited in the CSVs. Several mid-range scores (`google/gemma-2-2b-it` T3/many-shot, `mistralai/Mistral-Nemo-Instruct-2407` T2/few-shot, both Qwen3 models' T2 across most modes) reflect the model ignoring a fixed time window (the 2-3 PM meeting or the 1-3 PM class) rather than a formatting failure — see each row's `Evaluation Notes` for the specific evidence. Scoring was performed by reading each of the 60 raw responses directly against the Section 4 rubric; it has not yet been cross-checked by a second team member.


## Stage 3 — Shivani — Local LLM Integration (Models 11–15)

Models 11–15 use the **same fixed inputs (T1–T3), shared system instruction, and four prompt modes** (zero / one / few / many-shot) as Stage 2, through the same `test_model_prompt_modes` runner, so the 15-model comparison stays fair.

| # | Model | Category | Planned environment |
|---|---|---|---|
| 11 | `HuggingFaceTB/SmolLM2-360M-Instruct` | Tiny (<0.5B) | Local laptop, Transformers CPU |
| 12 | `Qwen/Qwen2.5-1.5B-Instruct` | Small (0.5B–1.5B) | Local laptop, Transformers CPU |
| 13 | `Qwen/Qwen2.5-3B-Instruct` | Medium (>1.5B–3B) | Google Colab T4, FP16 |
| 14 | `Qwen/Qwen2.5-7B-Instruct` | Large (>3B–7B) | Google Colab T4, 4-bit NF4 (FP16 ≈ 15 GB does not fit a 14.5 GB T4) |
| 15 | `Qwen/Qwen2.5-14B-Instruct` | Very Large (>7B) | Google Colab T4, 4-bit NF4 (FP16 ≈ 29 GB) |

**How to run.** Execution is disabled by default so that *Run All* never re-downloads a model or overwrites an already-scored CSV. Models 11–12: set `STAGE3_LOCAL_RUN` in the local-run cell. Models 13–15: open the notebook in Colab with a GPU runtime, run the bootstrap cell first, then the setup cells it lists, then set `RUN_COLAB_STAGE3 = True` in the Colab cell and run **one model per session**. Downloaded CSVs are copied into `llm-test/results/`.

Results are written to `llm-test/results/model_N_results.csv` and scored manually against the Section 4 rubric. Environments differ (local CPU vs. Colab GPU vs. 4-bit quantization), so response times are **not** a controlled speed comparison.

In [ ]:
# Stage 3: Local run for the two smallest Stage 3 models (11 and 12) on a laptop.
# Models 13-15 run in Colab (see the cells below).
# Execution is disabled by default so "Run All" never re-downloads/reruns a model
# or overwrites an already-scored CSV.

SHIVANI_MODELS = {
    11: "HuggingFaceTB/SmolLM2-360M-Instruct",
    12: "Qwen/Qwen2.5-1.5B-Instruct",
    13: "Qwen/Qwen2.5-3B-Instruct",     # Colab, FP16
    14: "Qwen/Qwen2.5-7B-Instruct",     # Colab, 4-bit NF4 (FP16 ~15 GB does not fit a T4)
    15: "Qwen/Qwen2.5-14B-Instruct",    # Colab, 4-bit NF4 (FP16 ~29 GB)
}

STAGE3_LOCAL_RUN = []        # e.g. [11], then [12]. Run one model at a time.
OVERWRITE_RESULTS = False    # keep False: protects manually scored CSVs

for model_no in STAGE3_LOCAL_RUN:
    model_id = SHIVANI_MODELS[model_no]
    out = RESULTS_DIR / f"model_{model_no}_results.csv"
    if out.exists() and not OVERWRITE_RESULTS:
        print(f"Skipping Model {model_no}: {out.name} already exists (set OVERWRITE_RESULTS = True to regenerate).")
        continue
    print(f"Running Model {model_no} — {model_id}")
    df = test_model_prompt_modes(model_id, test_cases)
    df.to_csv(out, index=False)
    print("Saved:", out)
    print(df[["Model", "Test ID", "Prompt Mode", "Status", "Response Time (s)"]])

In [2]:
# Stage 3: Colab bootstrap — run this FIRST, and ONLY in Google Colab.
# It clones the repository so the notebook's results-directory check finds llm-test/results/.
# Then run these cells (in this order): the portable model loader (first cell of the notebook),
# "Portable results directory", "Fixed CozyDay Test Inputs", "Shared Baseline Prompt",
# "Few-shot example pool", "Prompt-method comparison runner", and the Stage 3 local-run cell
# (it only defines SHIVANI_MODELS when STAGE3_LOCAL_RUN is empty). Do NOT use Run All.
import os, sys, subprocess

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/3_CozyDay"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/AshleyZhou-31/3_CozyDay.git", "/content/3_CozyDay"],
            check=True,
        )
    os.chdir("/content/3_CozyDay/llm-test")
    print("Colab working directory:", os.getcwd())
else:
    print("Not running in Google Colab; nothing to do.")

Not running in Google Colab; nothing to do.


In [26]:
# Stage 3: Colab runner for Models 13-15 (Qwen2.5-3B / 7B / 14B Instruct).
# Requires a CUDA GPU runtime (Runtime > Change runtime type > T4 GPU).
# Model 13 runs in FP16. Models 14 and 15 use 4-bit NF4 quantization (bitsandbytes) forced fully
# onto the GPU (device_map={"": 0}), the same configuration used for Model 10, because their FP16
# weights do not fit a free-tier T4 (~14.5 GB VRAM).
# Run ONE model per session: set COLAB_MODEL_NO, run, then Runtime > Restart session before the next.

RUN_COLAB_STAGE3 = False
COLAB_MODEL_NO = 13      # 13, 14 or 15

if RUN_COLAB_STAGE3:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43", "accelerate"], check=True)

    import torch, time, gc, pandas as pd
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("This cell requires a CUDA GPU in Colab. Runtime > Change runtime type > GPU.")
    if COLAB_MODEL_NO not in (13, 14, 15):
        raise ValueError("COLAB_MODEL_NO must be 13, 14 or 15.")

    out = RESULTS_DIR / f"model_{COLAB_MODEL_NO}_results.csv"
    if out.exists() and not OVERWRITE_RESULTS:
        raise FileExistsError(f"{out.name} already exists. Set OVERWRITE_RESULTS = True only if you intend to regenerate it.")

    selected = SHIVANI_MODELS[COLAB_MODEL_NO]
    tok = AutoTokenizer.from_pretrained(selected, trust_remote_code=False)
    if COLAB_MODEL_NO == 13:
        kwargs = dict(dtype=torch.float16, trust_remote_code=False, low_cpu_mem_usage=True)
        quant_label = "None (FP16)"
    else:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        kwargs = dict(quantization_config=bnb_config, device_map={"": 0}, trust_remote_code=False, low_cpu_mem_usage=True)
        quant_label = "4-bit NF4 (bitsandbytes)"
    mdl = AutoModelForCausalLM.from_pretrained(selected, **kwargs)
    if "device_map" not in kwargs:
        mdl = mdl.to("cuda")
    mdl.eval()

    rows = []
    try:
        for mode_name, n_shots in PROMPT_MODES.items():
            for case in test_cases:
                try:
                    prompt = build_prompt_shots(tok, case["input"], n_shots=n_shots)
                    tokens = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
                    torch.cuda.synchronize(); t0 = time.perf_counter()
                    with torch.inference_mode():
                        output = mdl.generate(**tokens, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
                    torch.cuda.synchronize()
                    elapsed = time.perf_counter() - t0
                    new_tokens = output[0][tokens["input_ids"].shape[1]:]
                    rows.append({
                        "Model": selected, "Test ID": case["test_id"], "Status": "Success",
                        "Response": tok.decode(new_tokens, skip_special_tokens=True),
                        "Response Time (s)": round(elapsed, 2), "Generated Tokens": len(new_tokens),
                        "Reached Token Limit": len(new_tokens) >= 256, "Error": "",
                        "Instruction Adherence": pd.NA, "Output Quality": pd.NA,
                        "Formatting Consistency": pd.NA, "Total Score": pd.NA,
                        "Quantization": quant_label, "Backend": "Transformers / Colab GPU",
                        "Prompt Mode": mode_name, "Evaluation Notes": "Pending manual scoring",
                    })
                    print(case["test_id"], mode_name, round(elapsed, 2), "seconds")
                except Exception as error:
                    rows.append({
                        "Model": selected, "Test ID": case["test_id"], "Status": "Failed",
                        "Response": "", "Response Time (s)": None, "Generated Tokens": None,
                        "Reached Token Limit": None, "Error": str(error),
                        "Instruction Adherence": pd.NA, "Output Quality": pd.NA,
                        "Formatting Consistency": pd.NA, "Total Score": pd.NA,
                        "Quantization": quant_label, "Backend": "Transformers / Colab GPU",
                        "Prompt Mode": mode_name, "Evaluation Notes": "Generation failed.",
                    })
                    print(case["test_id"], mode_name, "FAILED:", error)
    finally:
        del mdl, tok
        gc.collect()
        torch.cuda.empty_cache()

    pd.DataFrame(rows, columns=BASELINE_COLUMNS).to_csv(out, index=False)
    print("Saved:", out)
    try:
        from google.colab import files
        files.download(str(out))   # then copy the CSV into llm-test/results/ in the repository
    except ImportError:
        pass
else:
    print("Colab run not started. Set RUN_COLAB_STAGE3 = True inside an actual Colab GPU runtime.")

Colab run not started. Set RUN_COLAB_STAGE3 = True inside an actual Colab GPU runtime.


## Stage 3 — Results, Final Model Selection, and Handoff

**Completed:** All 5 assigned models (slots 11–15) ran successfully across all 4 prompt modes and all 3 shared test cases — 60/60 generations, `Status: Success`, zero errors. Models 11–12 ran locally via Transformers on a 16 GB Apple-silicon Mac (CPU path of the shared harness). Models 13–15 ran in Google Colab on a Tesla T4: Model 13 in FP16, Models 14 and 15 in 4-bit NF4 (`bitsandbytes`) because their FP16 weights (~15 GB and ~29 GB) do not fit a free-tier T4. All 60 responses were scored against the Section 4 rubric (0–5 each for Instruction Adherence, Output Quality, Formatting Consistency; max 15 per generation, 180 per model) and written to `llm-test/results/model_11_results.csv` through `model_15_results.csv` with an evaluation note per row. The unedited model responses are preserved in the CSVs.

### Per-model totals (3 test cases × 4 prompt modes; max 180)

| Model | Total /180 | Zero | One | Few | Many | Token-limit hits | Notes |
|---|---:|---:|---:|---:|---:|---:|---|
| 11 `HuggingFaceTB/SmolLM2-360M-Instruct` | 87 | 9 | 25 | 28 | 25 | 1 | Often picks the wrong priority or drops the `Priority/Schedule/Reason` headings; zero-shot is mostly unstructured prose |
| 12 `Qwen/Qwen2.5-1.5B-Instruct` | 110 | 24 | 30 | 26 | 30 | 1 | Solid on T1; frequently moves or omits the fixed meeting (T2) and class (T3) |
| 13 `Qwen/Qwen2.5-3B-Instruct` (FP16) | 137 | 33 | 37 | 37 | 30 | 0 | Perfect T2 in one-shot and few-shot; T3 repeatedly overlaps the 1–3 PM class; no truncation |
| 14 `Qwen/Qwen2.5-7B-Instruct` (4-bit) | 137 | 32 | 30 | 38 | 37 | 3 | Good T1/T2 in few/many-shot; T3 truncates before Reason or misallocates study time |
| 15 `Qwen/Qwen2.5-14B-Instruct` (4-bit) | 151 | 36 | 35 | 41 | 39 | 3 | Best of this slot: correct headings and constraints; loses points to 256-token truncation and over-allocated assignment time |

### Prompt-method comparison for this slot

Summed over Models 11–15 (max 225 per mode): zero-shot **134**, one-shot **157**, few-shot **170**, many-shot **161**. Few-shot is again the strongest, matching the Stage 2 finding; zero-shot is clearly the weakest because the small models lose the three-heading format without an example.

### Recurring failure patterns

- **T3 and the fixed 1–3 PM class:** nearly every low T3 score comes from placing the presentation, emails, or the class itself at the wrong time.
- **T2 and the fixed 2–3 PM meeting:** smaller models omit, shift, or double-book the meeting; the assignment is sometimes scheduled after its 5 PM deadline.
- **Truncation:** 8 of 60 responses (Models 11, 12, 14, 15) hit the 256-token limit and lost the `Reason` section. A larger `max_new_tokens` or a shorter schedule instruction would likely help; it was left unchanged to keep all 15 models comparable.
- **Format failures:** Model 11 zero-shot ignores the heading template entirely; Model 12 zero-shot reorders or drops sections.

### Full 15-model comparison

| # | Model | Category | Score | Prompt modes | Notes |
|---:|---|---|---|---|---|
| 1 | SmolLM2-135M-Instruct | Tiny | 6/45 | single baseline | Stage 1 |
| 2 | Qwen2.5-0.5B-Instruct | Small | 13/45 | single baseline | Stage 1 |
| 3 | Qwen3-1.7B | Medium | 12/45 | single baseline | Stage 1; `<think>` text, token-limit hits |
| 4 | Phi-3.5-mini-instruct | Large | 29/45 | single baseline | Stage 1 |
| 5 | Qwen3-8B | Very Large | 34/45 | single baseline | Stage 1; CPU offloading |
| 6 | SmolLM-135M-Instruct | Tiny | 25/180 | 4 modes | Stage 2 |
| 7 | Qwen3-0.6B | Small | 118/180 | 4 modes | Stage 2 |
| 8 | gemma-2-2b-it | Medium | 155/180 | 4 modes | Stage 2; gated |
| 9 | Qwen3-4B | Large | 141/180 | 4 modes | Stage 2; needs `enable_thinking=False` |
| 10 | Mistral-Nemo-Instruct-2407 (4-bit) | Very Large | 157/180 | 4 modes | Stage 2; needs CUDA |
| 11 | SmolLM2-360M-Instruct | Tiny | 87/180 | 4 modes | Stage 3 |
| 12 | Qwen2.5-1.5B-Instruct | Small | 110/180 | 4 modes | Stage 3 |
| 13 | Qwen2.5-3B-Instruct | Medium | 137/180 | 4 modes | Stage 3; **selected** |
| 14 | Qwen2.5-7B-Instruct (4-bit) | Large | 137/180 | 4 modes | Stage 3 |
| 15 | Qwen2.5-14B-Instruct (4-bit) | Very Large | 151/180 | 4 modes | Stage 3 |

Models 1–5 were scored once per test case (max 45), while Models 6–15 were scored over four prompt modes (max 180); the two score scales are **not** directly comparable.

### Final selection for the CozyDay feature

**Selected: `Qwen/Qwen2.5-3B-Instruct` with few-shot prompting (3 examples).**

- **Quality:** 137/180, with a perfect T2 result in one-shot and few-shot and zero truncated generations. It is the highest-scoring model that is both publicly downloadable and small enough for a laptop.
- **Footprint:** about 3B parameters (roughly 6 GB in FP16), so it runs through the same CPU/Transformers path as the local harness without a GPU or quantization.
- **Access:** no Hugging Face login or license-acceptance step, unlike `google/gemma-2-2b-it`, so teammates and graders can run the feature from a fresh clone.
- **Simplicity:** no thinking-mode flag is required, unlike `Qwen3-4B`.

**Trade-offs and alternatives:** Mistral-Nemo (157), gemma-2-2b-it (155), Qwen2.5-14B (151), and Qwen3-4B (141) all scored higher. Mistral-Nemo and the 14B model need a CUDA GPU with 4-bit quantization; gemma-2-2b-it is gated; Qwen3-4B is about 4 points higher but larger and requires `enable_thinking=False`. If a team member's machine can run it, the model ID is a Django setting so `google/gemma-2-2b-it` can be swapped in without code changes. The 3B model is released under the **Qwen Research License** (stated on its model card), which is suitable for this coursework but should be rechecked before any commercial use.

### Known limitations

- Scores are manual and apply only to these recorded outputs (greedy decoding, `max_new_tokens=256`, three tests); they are a comparison on a small fixed set, not a general benchmark.
- Response times are **not** a controlled speed comparison: local CPU, Colab FP16, and Colab 4-bit are different environments.
- Eight responses were truncated by the 256-token limit and lost part of the `Reason` section.
- The 3B model still double-books the 1–3 PM class on some T3 prompts, so the Django feature should preprocess input and validate output rather than trust the model's schedule blindly.

In [4]:
# Stage 3 evaluation summary: Models 11-15 (reads the scored CSVs; runs no model)
import pandas as pd

stage3_rows, stage3_mode_rows = [], []
for model_no in range(11, 16):
    path = RESULTS_DIR / f"model_{model_no}_results.csv"
    df3 = pd.read_csv(path)
    stage3_rows.append({
        "Model No.": model_no,
        "Model": df3["Model"].iloc[0],
        "Generations": len(df3),
        "Scored": int(df3["Total Score"].notna().sum()),
        "Token-limit hits": int(df3["Reached Token Limit"].sum()),
        "Total / 180": int(df3["Total Score"].sum()),
    })
    by_mode = df3.groupby("Prompt Mode")["Total Score"].sum()
    stage3_mode_rows.append({"Model No.": model_no, **{m: int(by_mode[m]) for m in ["Zero-shot", "One-shot", "Few-shot", "Many-shot"]}})

display(pd.DataFrame(stage3_rows))
mode_df = pd.DataFrame(stage3_mode_rows).set_index("Model No.")
mode_df.loc["All five"] = mode_df.sum()
display(mode_df)

,Model No.,Model,Generations,Scored,Token-limit hits,Total / 180
0,11,HuggingFaceTB/SmolLM2-360M-Instruct,12,12,1,87
1,12,Qwen/Qwen2.5-1.5B-Instruct,12,12,1,110
2,13,Qwen/Qwen2.5-3B-Instruct,12,12,0,137
3,14,Qwen/Qwen2.5-7B-Instruct,12,12,3,137
4,15,Qwen/Qwen2.5-14B-Instruct,12,12,3,151


,Zero-shot,One-shot,Few-shot,Many-shot
Model No.,,,,
11,9,25,28,25
12,24,30,26,30
13,33,37,37,30
14,32,30,38,37
15,36,35,41,39
All five,134,157,170,161
